# Training Notebook

### Objectives:
- Create a random forest regressor based model
- Use scikit-learn
- Explore the data to find the most important deciders of weather the flight is delayed
- Graph these explorations
- Split the datasets
- Get decent accuracy with the validation dataset

In [62]:
# Imports
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import duckdb as ddb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from memory_profiler import memory_usage
from time import perf_counter

# Constants

# run the proj in backend. This is because locating via Path(__file__) does not work with notebooks
DUCKDB_PATH = Path().cwd().resolve().parents[2]/"data/duck_database.duckdb"

In [ ]:
# Grab the data and declare vars
con = ddb.connect(DUCKDB_PATH)
data_df = con.sql("""
    SELECT * FROM model_dataset LIMIT 120000
""").df()
con.close()

data_df.columns
data_df = data_df.dropna(axis=0)

x_numeric_features = ['pred_dep_time', 'pred_arr_time', 'pred_elapsed_time',
       'fl_distance', 'origin_weather_code',
       'origin_temperature_2m_max', 'origin_temperature_2m_min',
       'origin_apparent_temperature_max', 'origin_apparent_temperature_min',
       'origin_precipitation_sum', 'origin_rain_sum', 'origin_showers_sum',
       'origin_snowfall_sum', 'origin_cloud_cover_mean',
       'origin_wind_speed_10m_max', 'origin_wind_gusts_10m_max',
       'origin_wind_direction_10m_dominant', 'origin_pressure_msl_mean',
       'dest_weather_code', 'dest_temperature_2m_max',
       'dest_temperature_2m_min', 'dest_apparent_temperature_max',
       'dest_apparent_temperature_min', 'dest_precipitation_sum',
       'dest_rain_sum', 'dest_showers_sum', 'dest_snowfall_sum',
       'dest_cloud_cover_mean', 'dest_wind_speed_10m_max',
       'dest_wind_gusts_10m_max', 'dest_wind_direction_10m_dominant',
       'dest_pressure_msl_mean']
x_categorical_features = ['flight_date', 'origin', 'dest']

x_features = x_numeric_features + x_categorical_features

y = data_df["delay"]
X = data_df[x_features]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

### Data exploration
- With notes on matplotlab (not great with it)

In [64]:
# Correlations

numeric_features = data_df[x_numeric_features + ["delay"]].dropna(axis=0).select_dtypes(include="number") # this essentially creates the df necessary for the corr table. drops all that are null and ensures all are nums
delay_correlations = (
    numeric_features
    .corr(numeric_only=True)["delay"]
    .drop("delay")
    .dropna()
    .sort_values(key=lambda values: values.abs()) # makes (for ex) -.5 a greater value than 0.2 (good for graph)
)

fig, ax = plt.subplots(figsize=(11, max(8, 0.34 * len(delay_correlations)))) # creates the plot and the spaces for each row/ the plot
colors = ["#b45309" if value < 0 else "#0f766e" for value in delay_correlations]

ax.barh(delay_correlations.index, delay_correlations.values, color=colors) # draws a horizontal bar chart. Preyy self explanotiry if you look at vars
ax.axvline(0, color="#222222", linewidth=0.8) # adds the middle line to show the start for all charts
ax.set_title("Pre-exploration Pearson correlation with delay") # title of the 'set' (table)
ax.set_xlabel("Correlation with delay") # title of the x
ax.set_ylabel("Numeric feature") # title of the y
ax.grid(axis="x", alpha=0.25) # adds the other lines allong the points so things are visable (poor epxlination but i mean the mildly seethorugh lines on the table)

fig.tight_layout() # auto does spacing so it looks sweet
# plt.show() # showing it in the notbook
fig.savefig("figures/correlation_chart.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [65]:
# Distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(data_df["delay"].dropna(), bins=range(-60, 301, 10), color="slategray", edgecolor="white")

ax.set_title("Distribution of flight delays")
ax.set_xlabel("Delay (mins)")
ax.set_ylabel("Number of flights")

ax.set_xlim(-10, 151)

fig.tight_layout()
fig.savefig("figures/delay_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [66]:
# Missing values

missing = data_df[x_features + ["delay"]].isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(missing.index, missing.values)

ax.set_title("Missing values in data")
ax.set_xlabel("Fraction missing")
ax.set_ylabel("Feature")

fig.tight_layout()
fig.savefig("figures/missing_values.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# No missing values, good :)

### Training
Methods for accuracy improvments:
- Duplicate accurate indicators (shown in correlation img)
- Combine features, for example: snow + wind
- Train many models

In [67]:
def train_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), x_categorical_features),
            ("numeric", "passthrough", x_numeric_features),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", RandomForestRegressor(random_state=1, n_jobs=-1)),
        ]
    )
    model.fit(X_train, y_train)
    return model


# Run
start = perf_counter()
mem_usage, model = memory_usage(
    (train_model,),
    retval=True,
    interval=0.1,
    max_usage=False
)
end = perf_counter()

print(f"Training seconds took: {end - start}")
print(f"Peak RAM: {max(mem_usage)}")

preds = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("MSE:", mean_squared_error(y_test, preds))
print("RMSE:", mean_squared_error(y_test, preds) ** 0.5)
print("R2:", r2_score(y_test, preds))

Training seconds took: 219.3112905839953
Peak RAM: 1821.125
MAE: 17.104063517857142
MSE: 2209.8376460083473
RMSE: 47.008910283140445
R2: -0.079391156586327
